# 10 — 8/26 조회수 상승 디플레이트 (+ 8/25 라벨 제약 · 과거 기준선 보정)

**리더보드 결과**

| 제출 | 구성 | 리더보드 |
|---|---|---|
| 07 | 전주 라벨 역추론 (기준) | 0.5917 |
| 10_s4 | 07 + 8/26 이후 조회수 디플레이트 | 0.6015 |
| 10_s2 | s4 + 8/25 라벨 제약 피처 11 | 0.6150 |
| **10_s3** | **s2 + 과거 기준선 보정** | **0.6290** |
| 10_s5 | s4 + 과거 기준선 보정 | (미제출) |
| submit_21 | 0.35·s4 + 0.35·s3 + 0.30·submit_15 순위평균 (11_ensemble) | **0.6420** |

**핵심 발견**: 08-26 이후 업로드된 롱폼은 채널 평소 대비 log 조회수가 **+0.40 ~ +0.85(약 2.2배)** 높다.
08월 초 영상은 같은 날 튀지 않으므로 측정 방식 변화가 아니라 신규 영상의 실제 상승이다.
학습 기준일(07-26 ~ 08-05)에는 이 상승이 없어서, 07은 09-01 테스트 피처(최근 모멘텀 등)가 부풀려진 채로 예측했다.
→ 상승분을 업로드일별로 빼서(디플레이트) 학습과 같은 기준으로 맞춘다.

> ⚠ **운영진 확인 필요** — 07과 마찬가지로 공식 `train_labels.csv`의 **08-25 라벨**(09-01 이후 영상 성과로 계산됨)을 피처로 쓴다.

In [1]:
# ===== 라이브러리 =====
import os, glob, warnings
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import average_precision_score
from IPython.display import display

warnings.filterwarnings('ignore')

# ===== 경로 · 상수 =====
RAW  = os.environ.get('EPOCH_RAW', '../data/raw/epoch_data')
SUB  = os.environ.get('EPOCH_SUB', '../submissions')
D, PREV = '2026-09-01', '2026-08-25'       # 본선 기준일 / 전주 공식 라벨 기준일
FUT, FUT_PREV = 26, 18                     # 본선 / 공식 라벨 정답 구간 길이 (일)
EVAL_DATES  = ['2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04', '2026-08-05']
TRAIN_DATES = pd.date_range('2026-07-26', '2026-08-05', freq='D').strftime('%Y-%m-%d').tolist()

# ===== DuckDB: CSV를 파일 이름 그대로 뷰로 등록 =====
con = duckdb.connect()
con.sql("SET memory_limit = '4GB'")
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    con.sql(f"CREATE OR REPLACE VIEW {os.path.splitext(os.path.basename(p))[0]} AS "
            f"SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

## 1. 8/26 이후 조회수 상승 확인 · 업로드일별 상승분 s(day)

채널 평소 수준 = 07-26 ~ 08-19 업로드 롱폼의 log view_5d 중앙값. 업로드일별로 (영상 log view_5d − 채널 평소 수준)의 중앙값을 본다.

In [2]:
daily = con.sql("""
    WITH x AS (SELECT channel_id, published_at, LN(1 + view_5d) AS lv FROM video_5d_views
               WHERE is_shorts = 0 AND view_5d IS NOT NULL),
    c AS (SELECT channel_id, MEDIAN(lv) AS cm FROM x
          WHERE published_at >= TIMESTAMP '2026-07-26' AND published_at < TIMESTAMP '2026-08-20' GROUP BY 1)
    SELECT CAST(published_at AS DATE) AS 업로드일, COUNT(*) AS 영상수, ROUND(MEDIAN(lv - cm), 3) AS 평소대비_log
    FROM x JOIN c USING (channel_id)
    WHERE published_at >= TIMESTAMP '2026-08-18' GROUP BY 1 ORDER BY 1""").df()
display(daily)

# 디플레이트에 쓸 상승분: 08-26 이후 업로드일만 (그 전은 0)
shift = daily[daily['업로드일'] >= pd.Timestamp('2026-08-26')].rename(columns={'업로드일': 'dd', '평소대비_log': 's'})[['dd', 's']]
# 정밀도를 위해 반올림 전 값으로 다시 계산
shift = con.sql("""
    WITH x AS (SELECT channel_id, published_at, LN(1 + view_5d) AS lv FROM video_5d_views WHERE is_shorts = 0 AND view_5d IS NOT NULL),
    c AS (SELECT channel_id, MEDIAN(lv) AS cm FROM x
          WHERE published_at >= TIMESTAMP '2026-07-26' AND published_at < TIMESTAMP '2026-08-20' GROUP BY 1)
    SELECT CAST(published_at AS DATE) AS dd, MEDIAN(lv - cm) AS s FROM x JOIN c USING (channel_id)
    WHERE published_at >= TIMESTAMP '2026-08-26' GROUP BY 1 ORDER BY 1""").df()
con.register('shift_tbl', shift)
shift.round(3)

,업로드일,영상수,평소대비_log
0,2026-08-18,439,0.000
1,2026-08-19,481,-0.014
2,2026-08-20,483,-0.046
3,2026-08-21,488,0.017
4,2026-08-22,497,-0.015
5,2026-08-23,448,0.075
6,2026-08-24,449,0.073
7,2026-08-25,477,0.148
8,2026-08-26,492,0.401
9,2026-08-27,507,0.763


,dd,s
0,2026-08-26,0.401
1,2026-08-27,0.763
2,2026-08-28,0.788
3,2026-08-29,0.829
4,2026-08-30,0.847
5,2026-08-31,0.820


## 2. view_5d 테이블 — `full`(원 값) / `defl`(상승분 제거)

학습 기준일(≤ 08-05)의 피처에는 08-26 이후 영상이 없으므로 두 모드가 같다. 차이는 **09-01 테스트 피처**와 **08-01 이후 기준일의 미래 라벨**에서만 생긴다.

In [3]:
_V5 = {}
def v5_view(Dref, mode):
    """기준일 이전 view_5d 테이블 (mode='defl'이면 08-26 이후 영상의 조회수·좋아요·댓글을 상승분만큼 낮춤)"""
    key = (Dref, mode)
    if key in _V5:
        return _V5[key]
    name = f"v5_{mode}_{Dref.replace('-', '')}"
    if mode == 'defl':
        cols = """EXP(LN(1 + f.view_5d)    - COALESCE(s.s, 0)) - 1 AS view_5d,
                  EXP(LN(1 + f.like_5d)    - COALESCE(s.s, 0)) - 1 AS like_5d,
                  EXP(LN(1 + f.comment_5d) - COALESCE(s.s, 0)) - 1 AS comment_5d"""
    else:   # 원 값 그대로
        cols = "f.view_5d, f.like_5d, f.comment_5d"
    con.sql(f"""CREATE OR REPLACE TEMP TABLE {name} AS
        SELECT f.video_id, f.channel_id, f.published_at, f.is_shorts, {cols}
        FROM video_5d_views f LEFT JOIN shift_tbl s ON CAST(f.published_at AS DATE) = s.dd
        WHERE f.view_5d IS NOT NULL AND f.published_at < TIMESTAMP '{Dref}'""")
    _V5[key] = name
    return name

## 3. 피처 (07과 같은 31개)

In [4]:
def make_features(Dref, mode):
    """기준일 이전 정보만으로 채널별 피처 23개 (07과 동일, view_5d는 v5_view에서만 읽음)"""
    v = v5_view(Dref, mode)
    f = con.sql(f"""
    WITH lv AS (   -- 롱폼 view_5d + 며칠 전 영상인지
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM {v} WHERE is_shorts = 0),
    v5 AS (        -- 롱폼 성과 요약
        SELECT channel_id, COUNT(*) AS n_long_5d, MEDIAN(view_5d) AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)   AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)  AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14) AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))          AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))       AS comment_rate,
               STDDEV(LN(1 + view_5d))                       AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)           AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)           AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)        AS trend_slope,
               MIN(days_ago)                                 AS days_since_meas
        FROM lv GROUP BY 1),
    sv AS (SELECT channel_id, MEDIAN(view_5d) AS short_med FROM {v} WHERE is_shorts = 1 GROUP BY 1),
    up AS (        -- 업로드 활동 (업로드 즉시 알 수 있으므로 published_at < Dref)
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0) AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0 AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1),
    games AS (     -- Steam 게임 매핑 수
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1)
    SELECT * FROM up LEFT JOIN v5 USING (channel_id) LEFT JOIN sv USING (channel_id) LEFT JOIN games USING (channel_id)
    """).df()
    f['n_long_5d'] = f.n_long_5d.fillna(0)
    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f


def add_prev_features(df, Dref, prev_labels, mode):
    """전주(D−7) 라벨 + 전주 정답 구간 중 이미 아는 부분 [D−7, D)의 성과 (07 역추론 피처 8개)"""
    v = v5_view(Dref, mode)
    P = (pd.Timestamp(Dref) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    pl = prev_labels[['channel_id', 'target']].rename(columns={'target': 'prev'})
    pp = con.sql(f"""SELECT channel_id, MEDIAN(view_5d) AS prev_past FROM {v}
                     WHERE is_shorts = 0 AND published_at < TIMESTAMP '{P}' GROUP BY 1""").df()
    kn = con.sql(f"""SELECT channel_id, COUNT(*) AS n_known, MEDIAN(view_5d) AS known_med FROM {v}
                     WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{P}' GROUP BY 1""").df()
    df = df.merge(pl, on='channel_id', how='left').merge(pp, on='channel_id', how='left').merge(kn, on='channel_id', how='left')
    df['n_known']        = df.n_known.fillna(0)
    df['has_prev']       = df.prev.notna().astype(int)
    df['prev_f']         = df.prev.fillna(0.2)
    df['log_prev_past']  = np.log1p(df.prev_past)
    df['known_rel_prev'] = np.log1p(df.known_med) - df.log_prev_past
    df['known_rel_now']  = np.log1p(df.known_med) - df.log_past_med
    df['prev_x_known']   = df.prev_f * (1 - df.known_rel_prev.fillna(0).clip(-3, 3) / 3)
    df['prev_x_lowpast'] = df.prev_f * (-df.log_past_med)
    return df


F23 = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
       'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
       'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
       'log_short_med_rel', 'log_n_long_5d']
PREVF = ['prev_f', 'has_prev', 'log_prev_past', 'n_known', 'known_rel_prev', 'known_rel_now', 'prev_x_known', 'prev_x_lowpast']
PAIR = F23 + PREVF

## 4. 8/25 라벨 제약 피처 (s2 · s3에서 사용)

전주 라벨 = 1 ⇔ 전주 정답 구간 영상 중앙값 ≥ (전주 과거 수준 + 임계값).
이미 아는 부분 [D−7, D)의 영상으로 임계값을 추정하고(학습·테스트 같은 방식), "D 이후 높은 영상이 최소 몇 개 필요했는지"를 계산한다.

In [5]:
CUR_MODE = 'full'   # known_logs가 읽을 모드 (build / test_rows에서 설정)

def known_logs(Dref):
    """[D−7, D) 롱폼 log view_5d 배열 (채널별)"""
    P = (pd.Timestamp(Dref) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    v = v5_view(Dref, CUR_MODE)
    k = con.sql(f"SELECT channel_id, LN(1 + view_5d) AS lv FROM {v} WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{P}'").df()
    return k.groupby('channel_id').lv.apply(np.array)


def est_threshold(df, K):
    """전주 라벨을 가장 잘 가르는 임계값 t: label ≈ [median(K) − log_prev_past ≥ t] (균형 정확도 최대)"""
    m = df.channel_id.map(lambda c: np.median(K[c]) if c in K.index else np.nan) - df.log_prev_past
    ok = df.prev.notna() & m.notna() & (df.channel_id.map(lambda c: len(K[c]) if c in K.index else 0) >= 3)
    y, x = df.prev[ok].values, m[ok].values
    cand = np.quantile(x, np.linspace(0.5, 0.97, 95))
    return max(cand, key=lambda t: ((x >= t) & (y == 1)).sum() / max((y == 1).sum(), 1) + ((x < t) & (y == 0)).sum() / max((y == 0).sum(), 1))


def add_cons(df, Dref):
    K = known_logs(Dref)
    t = est_threshold(df, K)
    lvl = t + df.log_prev_past                                  # 전주 라벨 = 1이 되려면 넘어야 할 중앙값(log)
    arr = df.channel_id.map(lambda c: K[c] if c in K.index else np.empty(0))
    a = np.array([(k >= l).sum() if pd.notna(l) else 0 for k, l in zip(arr, lvl)])
    b = np.array([(k < l).sum() if pd.notna(l) else 0 for k, l in zip(arr, lvl)])
    prev = df.prev.values
    out = pd.DataFrame(index=df.index)
    out['c_thr'] = t
    out['c_margin'] = df.channel_id.map(lambda c: np.median(K[c]) if c in K.index else np.nan) - lvl
    out['c_a'], out['c_b'] = a, b
    out['c_need_up'] = np.maximum(0, b - a)                     # 라벨=1이려면 D 이후 높은 영상이 최소 몇 개
    out['c_need_dn'] = np.maximum(0, a - b + 1)                 # 라벨=0이려면 D 이후 낮은 영상(또는 무업로드)
    out['c_up'] = np.where(prev == 1, out.c_need_up, 0)
    out['c_dn'] = np.where(prev == 0, out.c_need_dn, 0)
    out['c_surprise'] = np.where(prev == 1, -out.c_margin, np.where(prev == 0, out.c_margin, np.nan))
    out['c_lvl_now'] = lvl - df.log_past_med
    out['c_lb'] = np.where(prev == 1, out.c_lvl_now, np.nan)
    out['c_ub'] = np.where(prev == 0, out.c_lvl_now, np.nan)
    return pd.concat([df, out], axis=1)

CONS = ['c_margin', 'c_a', 'c_b', 'c_need_up', 'c_need_dn', 'c_up', 'c_dn', 'c_surprise', 'c_lvl_now', 'c_lb', 'c_ub']

## 5. 라벨 · 학습/테스트 데이터

In [6]:
def labels_g(Dref, fut, defl=False):
    """공식 규칙 라벨 + growth. defl=True면 08-26 이후 영상을 디플레이트한 값으로 계산"""
    val = "EXP(LN(1 + f.view_5d) - COALESCE(s.s, 0)) - 1" if defl else "f.view_5d"
    return con.sql(f"""
        WITH src AS (SELECT f.channel_id, f.published_at, {val} AS v FROM video_5d_views f
                     LEFT JOIN shift_tbl s ON CAST(f.published_at AS DATE) = s.dd
                     WHERE f.is_shorts = 0 AND f.view_5d IS NOT NULL),
        agg AS (SELECT channel_id, MEDIAN(v) FILTER (WHERE published_at < TIMESTAMP '{Dref}') ps,
                       COUNT(v) FILTER (WHERE published_at < TIMESTAMP '{Dref}') n,
                       COALESCE(MEDIAN(v) FILTER (WHERE published_at >= TIMESTAMP '{Dref}'
                                AND published_at < TIMESTAMP '{Dref}' + INTERVAL {fut} DAY), 0) fs
                FROM src GROUP BY 1),
        sc AS (SELECT channel_id, LN(1 + fs) - LN(1 + ps) AS growth FROM agg WHERE n >= 3 AND ps >= 100)
        SELECT channel_id, growth, CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target FROM sc""").df()


def build(mode, defl_label):
    """학습 기준일(07-26 ~ 08-05)별: 26일 라벨 + 피처 + 전주(D−7) 18일 라벨 역추론 + 제약 피처"""
    global CUR_MODE
    CUR_MODE = mode
    parts = []
    for d in TRAIN_DATES:
        P = (pd.Timestamp(d) - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
        lab = labels_g(d, FUT, defl_label).assign(ref_date=d)
        raw = labels_g(d, FUT, False)[['channel_id', 'target']].rename(columns={'target': 'target_raw'})   # 평가용 공식 규칙 라벨
        df = lab.merge(raw, on='channel_id', how='left').merge(make_features(d, mode), on='channel_id', how='left')
        df = add_prev_features(df, d, labels_g(P, FUT_PREV, defl_label), mode)
        parts.append(add_cons(df, d))
    return pd.concat(parts, ignore_index=True)


# 공식 08-25 라벨 · 채점 대상
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
PREV_OFF = tl[tl.row_id.str.endswith(PREV)][['channel_id', 'target']]
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]

def test_rows(mode):
    """09-01 테스트 행: 학습과 같은 함수로 피처 생성"""
    global CUR_MODE
    CUR_MODE = mode
    t = sample[['row_id', 'channel_id']].merge(make_features(D, mode), on='channel_id', how='left')
    t = add_prev_features(t, D, PREV_OFF, mode)
    return add_cons(t, D)

TR_dl = build('defl', True)       # 학습: 디플레이트 피처 + 디플레이트 라벨
TE_full, TE_defl = test_rows('full'), test_rows('defl')
print('학습 행:', len(TR_dl), '| 테스트 행:', len(TE_defl))

# 디플레이트 효과: 테스트 피처가 학습 분포로 돌아오는지
tr05 = TR_dl[TR_dl.ref_date == '2026-08-05']
display(pd.DataFrame({'학습 08-05': tr05[['mom_7', 'known_rel_prev', 'c_margin']].median(),
                      '테스트 원 값(07)': TE_full[['mom_7', 'known_rel_prev', 'c_margin']].median(),
                      '테스트 디플레이트': TE_defl[['mom_7', 'known_rel_prev', 'c_margin']].median()}).round(3))
print('전주 라벨 임계값 추정 — 테스트 원 값:', round(TE_full.c_thr.iloc[0], 3), '| 디플레이트:', round(TE_defl.c_thr.iloc[0], 3))

학습 행: 5102 | 테스트 행: 694


,학습 08-05,테스트 원 값(07),테스트 디플레이트
mom_7,0.000,0.531,0.000
known_rel_prev,0.000,0.663,-0.004
c_margin,-0.284,-0.439,-0.393


전주 라벨 임계값 추정 — 테스트 원 값: 1.079 | 디플레이트: 0.37


## 6. 모델 (07과 동일) · 검증

In [7]:
def hgb(seed):
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=30,
                                          l2_regularization=1.0, random_state=seed)

def logreg():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=3000, C=0.1, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, float))
    return rankdata(x.fillna(x.median())) / len(x)

def predict(tr, te, feats):
    """HGB 5시드 평균 순위 + 로지스틱 순위의 평균 → (순위 점수, HGB 확률)"""
    p_hgb = np.mean([hgb(s).fit(tr[feats], tr.target).predict_proba(te[feats])[:, 1] for s in range(5)], axis=0)
    p_lr = logreg().fit(tr[feats], tr.target).predict_proba(te[feats])[:, 1]
    return (rank01(p_hgb) + rank01(p_lr)) / 2, p_hgb

def validate_raw(data, feats):
    """평가일마다 (평가일 − 4일) 이하로 학습 → 공식 규칙 라벨(target_raw)로 PR-AUC"""
    out = {}
    for td in EVAL_DATES:
        te = data[data.ref_date == td]
        tr = data[data.ref_date <= (pd.Timestamp(td) - pd.Timedelta(days=4)).strftime('%Y-%m-%d')]
        s, _ = predict(tr, te, feats)
        out[td[5:]] = average_precision_score(te.target_raw, s)
    out['평균'] = np.mean(list(out.values()))
    return out

display(pd.DataFrame({'s4 (피처 31)': validate_raw(TR_dl, PAIR),
                      's2 (+ 제약 11)': validate_raw(TR_dl, PAIR + CONS)}).T.round(3))

,08-01,08-02,08-03,08-04,08-05,평균
s4 (피처 31),0.557,0.562,0.543,0.587,0.569,0.563
s2 (+ 제약 11),0.575,0.592,0.566,0.585,0.578,0.579


## 7. 제출 파일

In [8]:
SUB07 = pd.read_csv(f'{SUB}/submit_07_prevlabel_inversion_hgb.csv')

def save(score, tie, fname):
    """동점 깨기 → 0~1 정규화 → sample 순서 → 제출 가이드 §3 검사 → 저장"""
    s = np.asarray(score, float) + 1e-6 * rank01(tie)
    s = (s - s.min()) / (s.max() - s.min())
    sub = sample[['row_id']].merge(pd.DataFrame({'row_id': TE_full.row_id, 'prediction': s}), on='row_id', how='left')
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694
    assert set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and sub.prediction.between(0, 1).all()
    sub.to_csv(f'{SUB}/{fname}', index=False)
    m = sub.merge(SUB07, on='row_id')
    print(fname, '| 07과 Spearman', round(spearmanr(m.prediction_x, m.prediction_y)[0], 3))

def growth_reg(TR, TE, feats):
    """디플레이트 공간 growth 회귀 (3시드 평균) — 기준선 보정용"""
    return np.mean([HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=300, min_samples_leaf=30,
                    random_state=k).fit(TR[feats], TR.growth.clip(-3, 3)).predict(TE[feats]) for k in range(3)], axis=0)

def adjust(s, p_hgb, mu, d_c):
    """과거 기준선 왜곡 d_c만큼 분류 logit과 회귀 μ를 낮춘 뒤 원 점수와 순위 평균"""
    lg = np.log(p_hgb / (1 - p_hgb))
    beta = np.cov(lg, mu)[0, 1] / np.var(mu)                    # growth 1단위당 logit 변화
    z1, z2 = lg - beta * np.nan_to_num(d_c), mu - np.nan_to_num(d_c)
    return (rank01(z1) + rank01(z2) + rank01(s)) / 3, z1

# 08-26 상승분이 09-01 과거 중앙값을 얼마나 부풀렸는지 (채널별)
d_c = (TE_full.log_past_med - TE_defl.log_past_med).values
print('과거 기준선 왜곡 d_c > 0.1 채널:', int((d_c > 0.1).sum()))

과거 기준선 왜곡 d_c > 0.1 채널: 201


In [9]:
# ===== 10_s4 (리더보드 0.629) — 07 피처 31 + 디플레이트 =====
s4, ph4 = predict(TR_dl, TE_defl, PAIR)
save(s4, ph4, 'submit_10_s4_07_defl.csv')

# ===== 10_s2 (0.615) — s4 + 제약 피처 11 =====
s2, ph2 = predict(TR_dl, TE_defl, PAIR + CONS)
save(s2, ph2, 'submit_10_s2_cons_defl.csv')

# ===== 10_s3 (0.611) — s2 + 과거 기준선 보정 =====
s3, z3 = adjust(s2, ph2, growth_reg(TR_dl, TE_defl, PAIR + CONS), d_c)
save(s3, z3, 'submit_10_s3_cons_defl_adj.csv')

# ===== 10_s5 — s4 + 과거 기준선 보정 =====
s5, z5 = adjust(s4, ph4, growth_reg(TR_dl, TE_defl, PAIR), d_c)
save(s5, z5, 'submit_10_s5_07_defl_adj.csv')

submit_10_s4_07_defl.csv | 07과 Spearman 0.933


submit_10_s2_cons_defl.csv | 07과 Spearman 0.893


submit_10_s3_cons_defl_adj.csv | 07과 Spearman 0.85


submit_10_s5_07_defl_adj.csv | 07과 Spearman 0.873


## 8. 정리

- **세 단계 모두 리더보드에서 효과**: 디플레이트(s4) +0.010 → 제약 피처(s2) +0.0135 → 과거 기준선 보정(s3) +0.014. 07 대비 누적 +0.037.
- 디플레이트는 테스트 피처 왜곡(최근 모멘텀 `mom_7` 0.53 → 0.00)을 없앴고, 제약 피처는 8/25 라벨을 임계값 기반으로 더 정확히 썼고, 기준선 보정은 8/26 급등이 부풀린 09-01 과거 중앙값(201채널)을 바로잡았다.
- s3를 팀원 모델(submit_15)·s4와 순위평균한 submit_21이 0.6420으로 최고점.
- 참고: 같은 구조에서 모델만 CatBoost로 바꾼 버전은 백테스트 +0.04였지만 리더보드에서는 하락(submit_31 0.6004). 학습 기간에 없던 급등 때문에 백테스트로 모델 종류를 고르면 안 된다.
